In [29]:
from pathlib import Path
import zipfile

import pandas as pd
import searoute as sr
from geopy.distance import great_circle
import coordinates as coord
import os
from core_modules import core_modules as core

In this code, I assemble the necessary data, clean it, conduct an exploratory data analysis, and then ...

# Data Collection

## Distance/Weight
United States Census has necessary information about distance of trade routes and weight of imports. Despite its name, the PORTHS6MM dataset contains information on land ports as well.


This code downloads the files from the website. Each of those files weighs hundreds of megabytes, but once it's downloaded, the command sees that they're inside the right directory and skips instead of downloading them again.

In [ ]:
import scripts.download_data as dd


for year in range(2022, 2026):
    dd.download(year, 12)


  downloading PORTHS6MM2212.ZIP from https://www.census.gov/trade/downloads/2022/Port/im_hs6_m/PORTHS6MM2212.ZIP ... HTTP 403 — file may not yet be released or URL has changed
  downloading PORTHS6MM2312.ZIP from https://www.census.gov/trade/downloads/2023/Port/im_hs6_m/PORTHS6MM2312.ZIP ... HTTP 403 — file may not yet be released or URL has changed
  downloading PORTHS6MM2412.ZIP from https://www.census.gov/trade/downloads/2024/Port/im_hs6_m/PORTHS6MM2412.ZIP ... HTTP 403 — file may not yet be released or URL has changed
  downloading PORTHS6MM2512.ZIP from https://www.census.gov/trade/downloads/2025/Port/im_hs6_m/PORTHS6MM2512.ZIP ... HTTP 403 — file may not yet be released or URL has changed


## Grid intensity

Grid intensity is important for manufacturing, but it's still relevant for supply chains. Source: [Ember Energy](https://ember-energy.org/latest-insights/global-electricity-review-2025/major-countries-and-regions/)



In [31]:
# gCO2/kWh - grid intensity is about CO2 released per unit of energy. Mostly about manufacturing, but still relevant
grid_intensity = {"china": 525, "mexico": 412, "s_korea": 390}


## Emission Factor
Ton-km emission factor is co2(tons)(tons)(tons)(tons)(tons)(tons)(tons) released per ton of commodity per kilometer transported. Relevant for transportation, depends on country's mix of transportation methods used. Because washing machines are transported mainly by trade vessels and trucks, these two are the main methods used in the calculation of the emission factor.

Lane-specific emission factors combine the IMO Fourth GHG Study global average with adjustments for typical vessel deployment on each lane (sourced from UNCTAD 2024 Chapter II) and feeder-megaship transshipment patterns documented in Notteboom & Rodrigue (2009).

Instead of country names, country codes were used as per [country.txt](https://www.census.gov/foreign-trade/schedules/c/country.txt) file on Census.gov.

Sources:
+ [US EPA SmartWay Carrier Emission Factors](epa.gov/smartway) - emission factor of Mexico-US trade routes
+ [New shipping routes highlight growing Asia-to-Mexico trade](https://www.freightwaves.com/news/new-shipping-routes-highlight-growing-asia-to-mexico-trade) - emission factor of Asia-Pacific trade routes
+ [Review of Maritime Transport 2024: Navigating Maritime Chokepoints](https://unctad.org/publication/review-maritime-transport-2024.) - GHG global average with country-specific adjustments for each trade route.
+ [Notteboom and Rodrigue](https://doi.org/10.1007/s10708-008-9210-4) - documents shipment patterns of feeder megaships

# Data Cleaning
Cleaning is the longest and the most important step of any data cycle. After all, without good data there can be no good results. Because this projects uses data from a wide variety of sources, this means dealing with many different formats, which may complicate the cleaning process even further.

In [32]:
# this defines the columns in the PORTHS6MM dataframe, which are to be converted to numbers
numerical_cols_asian = ["gen_val_mo", "air_val_mo", "air_swt_mo", "ves_val_mo", "ves_swt_mo", "cnt_val_mo", "cnt_swt_mo"]


## Ocean-based Distance/Weight Data (Asian Routes, Pre-Tariff — Dec 2024)

In [33]:
_snapshot = Path('data/snapshots/PORTHS6MM2412.parquet')
if _snapshot.exists():
    df_ocean_routes_2412 = pd.read_parquet(_snapshot)
else:
    with zipfile.ZipFile('data/original/PORTHS6MM2412.ZIP') as zf:
        with zf.open(zf.namelist()[0]) as f:
            df_ocean_routes_2412 = pd.read_fwf(
                f,
                colspecs=coord.colspecs_sea,
                names=coord.names_sea,
                dtype={c: str for c in coord.str_cols_sea},
            )
    _snapshot.parent.mkdir(exist_ok=True)
    df_ocean_routes_2412.to_parquet(_snapshot, index=False)


In [34]:
df_ocean_routes_2412.info()

<class 'pandas.DataFrame'>
RangeIndex: 1212596 entries, 0 to 1212595
Data columns (total 20 columns):
 #   Column       Non-Null Count    Dtype
---  ------       --------------    -----
 0   commodity    1212596 non-null  str  
 1   cty_code     1212596 non-null  str  
 2   dist_unlade  1212596 non-null  str  
 3   port_unlade  1212596 non-null  str  
 4   year         1212596 non-null  str  
 5   month        1212596 non-null  str  
 6   gen_val_mo   1212596 non-null  int64
 7   air_val_mo   1212596 non-null  int64
 8   air_swt_mo   1212596 non-null  int64
 9   ves_val_mo   1212596 non-null  int64
 10  ves_swt_mo   1212596 non-null  int64
 11  cnt_val_mo   1212596 non-null  int64
 12  cnt_swt_mo   1212596 non-null  int64
 13  gen_val_yr   1212596 non-null  int64
 14  air_val_yr   1212596 non-null  int64
 15  air_swt_yr   1212596 non-null  int64
 16  ves_val_yr   1212596 non-null  int64
 17  ves_swt_yr   1212596 non-null  int64
 18  cnt_val_yr   1212596 non-null  int64
 19  cnt_swt_yr 

In [35]:
df_ocean_routes_2412["port_full"] = df_ocean_routes_2412["dist_unlade"] + df_ocean_routes_2412["port_unlade"]

df_asian_2412 = df_ocean_routes_2412[df_ocean_routes_2412["cty_code"].isin(coord.asian_origins)].T.drop_duplicates().T
for col in numerical_cols_asian:
    df_asian_2412[col] = pd.to_numeric(df_asian_2412[col])

df_asian_wash_2412 = df_asian_2412[df_asian_2412["commodity"].isin(["845011", "845020"])].copy()
df_asian_wash_2412 = df_asian_wash_2412[df_asian_wash_2412["port_full"].isin(coord.us_ports)]
df_asian_wash_2412 = df_asian_wash_2412[df_asian_wash_2412["ves_swt_mo"] > 0]
df_asian_wash_2412["distance"] = df_asian_wash_2412.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_wash_2412["co2"] = df_asian_wash_2412.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

df_asian_wash_2412.to_csv("data/intermediate/PORTHS6MM_asian_wash_2412.csv")
df_asian_wash_2412.info()

<class 'pandas.DataFrame'>
Index: 79 entries, 799729 to 800031
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    79 non-null     object 
 1   cty_code     79 non-null     object 
 2   dist_unlade  79 non-null     object 
 3   port_unlade  79 non-null     object 
 4   year         79 non-null     object 
 5   month        79 non-null     object 
 6   gen_val_mo   79 non-null     int64  
 7   air_val_mo   79 non-null     int64  
 8   air_swt_mo   79 non-null     int64  
 9   ves_val_mo   79 non-null     int64  
 10  ves_swt_mo   79 non-null     int64  
 11  cnt_val_mo   79 non-null     int64  
 12  cnt_swt_mo   79 non-null     int64  
 13  gen_val_yr   79 non-null     object 
 14  air_val_yr   79 non-null     object 
 15  air_swt_yr   79 non-null     object 
 16  ves_val_yr   79 non-null     object 
 17  ves_swt_yr   79 non-null     object 
 18  cnt_val_yr   79 non-null     object 
 19  cnt_swt_yr   79 n

In [36]:
df_asian_wash_2412["port_full"].value_counts().sort_index(ascending=False)

port_full
5301    5
5201    3
4909    4
3002    5
3001    6
2904    3
2811    5
2709    6
2704    6
2002    1
1901    2
1803    4
1801    4
1703    6
1601    3
1401    4
1303    5
1003    6
0401    1
Name: count, dtype: int64

## Ocean-based Distance/Weight Data (Asian Routes, Post-Tariff — Dec 2025)

In [37]:
_snapshot = Path("data/snapshots/PORTHS6MM2512.parquet")
if _snapshot.exists():
    df_ocean_routes_2512 = pd.read_parquet(_snapshot)
else:
    with zipfile.ZipFile("data/original/PORTHS6MM2512.ZIP") as zf:
        with zf.open(zf.namelist()[0]) as f:
            df_ocean_routes_2512 = pd.read_fwf(
                f,
                colspecs=coord.colspecs_sea,
                names=coord.names_sea,
                dtype={c: str for c in coord.str_cols_sea},
            )
    _snapshot.parent.mkdir(exist_ok=True)
    df_ocean_routes_2512.to_parquet(_snapshot, index=False)

df_ocean_routes_2512["port_full"] = df_ocean_routes_2512["dist_unlade"] + df_ocean_routes_2512["port_unlade"]

df_asian_2512 = df_ocean_routes_2512[df_ocean_routes_2512["cty_code"].isin(coord.asian_origins)].T.drop_duplicates().T
for col in numerical_cols_asian:
    df_asian_2512[col] = pd.to_numeric(df_asian_2512[col])

df_asian_wash_2512 = df_asian_2512[df_asian_2512["commodity"].isin(["845011", "845020"])].copy()
df_asian_wash_2512 = df_asian_wash_2512[df_asian_wash_2512["port_full"].isin(coord.us_ports)]
df_asian_wash_2512 = df_asian_wash_2512[df_asian_wash_2512["ves_swt_mo"] > 0]
df_asian_wash_2512["distance"] = df_asian_wash_2512.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_wash_2512["co2"] = df_asian_wash_2512.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

df_asian_wash_2512.to_csv("data/intermediate/PORTHS6MM_asian_wash_2512.csv")
df_asian_wash_2512.info()

<class 'pandas.DataFrame'>
Index: 73 entries, 824153 to 824435
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    73 non-null     object 
 1   cty_code     73 non-null     object 
 2   dist_unlade  73 non-null     object 
 3   port_unlade  73 non-null     object 
 4   year         73 non-null     object 
 5   month        73 non-null     object 
 6   gen_val_mo   73 non-null     int64  
 7   air_val_mo   73 non-null     int64  
 8   air_swt_mo   73 non-null     int64  
 9   ves_val_mo   73 non-null     int64  
 10  ves_swt_mo   73 non-null     int64  
 11  cnt_val_mo   73 non-null     int64  
 12  cnt_swt_mo   73 non-null     int64  
 13  gen_val_yr   73 non-null     object 
 14  air_val_yr   73 non-null     object 
 15  air_swt_yr   73 non-null     object 
 16  ves_val_yr   73 non-null     object 
 17  ves_swt_yr   73 non-null     object 
 18  cnt_val_yr   73 non-null     object 
 19  cnt_swt_yr   73 n

In [38]:
df_asian_wash_2512["port_full"].value_counts()

port_full
1003    6
1703    6
2704    6
1303    5
1801    5
2709    5
3002    5
1401    4
2811    4
1601    4
1803    4
5201    4
5301    4
3001    3
4909    3
1901    2
1001    1
2002    1
2904    1
Name: count, dtype: int64

## Ocean-based Distance/Weight Data (Asian Routes, Control — HS 8528 TVs, Dec 2024)

In [39]:
df_asian_tv_2412 = df_asian_2412[df_asian_2412["commodity"].str.startswith("8528")].copy()
df_asian_tv_2412 = df_asian_tv_2412[df_asian_tv_2412["port_full"].isin(coord.us_ports)]
df_asian_tv_2412 = df_asian_tv_2412[df_asian_tv_2412["ves_swt_mo"] > 0]
df_asian_tv_2412["distance"] = df_asian_tv_2412.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_tv_2412["co2"] = df_asian_tv_2412.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

df_asian_tv_2412.to_csv("data/intermediate/PORTHS6MM_asian_tv_2412.csv")
df_asian_tv_2412.info()

<class 'pandas.DataFrame'>
Index: 116 entries, 953212 to 955952
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    116 non-null    object 
 1   cty_code     116 non-null    object 
 2   dist_unlade  116 non-null    object 
 3   port_unlade  116 non-null    object 
 4   year         116 non-null    object 
 5   month        116 non-null    object 
 6   gen_val_mo   116 non-null    int64  
 7   air_val_mo   116 non-null    int64  
 8   air_swt_mo   116 non-null    int64  
 9   ves_val_mo   116 non-null    int64  
 10  ves_swt_mo   116 non-null    int64  
 11  cnt_val_mo   116 non-null    int64  
 12  cnt_swt_mo   116 non-null    int64  
 13  gen_val_yr   116 non-null    object 
 14  air_val_yr   116 non-null    object 
 15  air_swt_yr   116 non-null    object 
 16  ves_val_yr   116 non-null    object 
 17  ves_swt_yr   116 non-null    object 
 18  cnt_val_yr   116 non-null    object 
 19  cnt_swt_yr   116

In [40]:
TvPortcodes2412 = df_asian_tv_2412["port_full"].value_counts().sort_index()
with pd.option_context('display.max_rows', None,
                       'display.max_columns', None,
                       'display.precision', 3,
                       ):
    print(TvPortcodes2412)

port_full
0401     2
1001     2
1003    11
1101     1
1303     2
1401     6
1501     1
1601     6
1703     9
1801     2
1803     1
1816     2
1901     2
2704    15
2709    15
2809     2
2811     7
3001     5
3002     8
3604     2
4909     3
5201     6
5203     1
5301     4
5310     1
Name: count, dtype: int64


## Ocean-based Distance/Weight Data (Asian Routes, Control — HS 8528 TVs, Dec 2025)

In [41]:
df_asian_tv_2512 = df_asian_2512[df_asian_2512["commodity"].str.startswith("8528")].copy()
df_asian_tv_2512 = df_asian_tv_2512[df_asian_tv_2512["port_full"].isin(coord.us_ports)]
df_asian_tv_2512 = df_asian_tv_2512[df_asian_tv_2512["ves_swt_mo"] > 0]
df_asian_tv_2512["distance"] = df_asian_tv_2512.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_tv_2512["co2"] = df_asian_tv_2512.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

df_asian_tv_2512.to_csv("data/intermediate/PORTHS6MM_asian_tv_2512.csv")
df_asian_tv_2512.info()

<class 'pandas.DataFrame'>
Index: 111 entries, 978494 to 981139
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    111 non-null    object 
 1   cty_code     111 non-null    object 
 2   dist_unlade  111 non-null    object 
 3   port_unlade  111 non-null    object 
 4   year         111 non-null    object 
 5   month        111 non-null    object 
 6   gen_val_mo   111 non-null    int64  
 7   air_val_mo   111 non-null    int64  
 8   air_swt_mo   111 non-null    int64  
 9   ves_val_mo   111 non-null    int64  
 10  ves_swt_mo   111 non-null    int64  
 11  cnt_val_mo   111 non-null    int64  
 12  cnt_swt_mo   111 non-null    int64  
 13  gen_val_yr   111 non-null    object 
 14  air_val_yr   111 non-null    object 
 15  air_swt_yr   111 non-null    object 
 16  ves_val_yr   111 non-null    object 
 17  ves_swt_yr   111 non-null    object 
 18  cnt_val_yr   111 non-null    object 
 19  cnt_swt_yr   111

Mexico's PORTHS6MM records do not carry a direct land-transport weight column — only `gen_val_mo` (total import value) and `ves_swt_mo` (vessel weight, zero for land crossings). To estimate land shipment weight I compute a \$/kg price coefficient from the subset of Mexico washing-machine rows that *did* arrive by vessel (`ves_swt_mo > 0`, e.g. Tampa and Puerto Rico), then apply it as `gen_swt_mo = gen_val_mo / washing_machine_price_coeff` for all rows.

In [42]:
# Price coefficient from PORTHS6MM vessel-shipped Mexico washing machines
_mex_ves = df_ocean_routes_2412[
    (df_ocean_routes_2412["cty_code"] == "2010") &
    (df_ocean_routes_2412["commodity"].isin(["845011", "845020"]))
].copy()
for col in ["ves_val_mo", "ves_swt_mo"]:
    _mex_ves[col] = pd.to_numeric(_mex_ves[col], errors="coerce").fillna(0)
_mex_ves = _mex_ves[_mex_ves["ves_swt_mo"] > 0]

washing_machine_price_coeff = _mex_ves["ves_val_mo"].mean() / _mex_ves["ves_swt_mo"].mean()
print(washing_machine_price_coeff, "$/kg")

4.694152410295217 $/kg


In [43]:
if not df_asian_tv_2412.empty and df_asian_tv_2412["ves_swt_mo"].sum() > 0:
    tv_price_coeff = df_asian_tv_2412["ves_val_mo"].mean() / df_asian_tv_2412["ves_swt_mo"].mean()
else:
    tv_price_coeff = 15.0  # fallback: ~$15/kg for TVs
print(tv_price_coeff, "$/kg")

17.303979051815883 $/kg


In [44]:
# Mexico washing machines — Dec 2024 (PORTHS6MM)
df_mex_wash_2412 = df_ocean_routes_2412[
    (df_ocean_routes_2412["cty_code"] == "2010") &
    (df_ocean_routes_2412["commodity"].isin(["845011", "845020"])) &
    (df_ocean_routes_2412["port_full"].isin(coord.us_ports))
].copy()

for col in ["gen_val_mo", "ves_val_mo", "ves_swt_mo"]:
    df_mex_wash_2412[col] = pd.to_numeric(df_mex_wash_2412[col], errors="coerce").fillna(0)

df_mex_wash_2412 = df_mex_wash_2412[df_mex_wash_2412["gen_val_mo"] > 0]

df_mex_wash_2412["gen_swt_mo"] = df_mex_wash_2412["ves_swt_mo"].where(
    df_mex_wash_2412["ves_swt_mo"] > 0,
    df_mex_wash_2412["gen_val_mo"] / washing_machine_price_coeff
)
df_mex_wash_2412["distance"] = df_mex_wash_2412.apply(
    core.compute_distance,
    args=(coord.mexico_start, coord.us_ports, False, None, "port_full"), axis=1)
df_mex_wash_2412["co2"] = df_mex_wash_2412.apply(core.compute_co2, args=["gen_swt_mo"], axis=1)
df_mex_wash_2412.info()

<class 'pandas.DataFrame'>
Index: 12 entries, 799677 to 799855
Data columns (total 24 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    12 non-null     str    
 1   cty_code     12 non-null     str    
 2   dist_unlade  12 non-null     str    
 3   port_unlade  12 non-null     str    
 4   year         12 non-null     str    
 5   month        12 non-null     str    
 6   gen_val_mo   12 non-null     int64  
 7   air_val_mo   12 non-null     int64  
 8   air_swt_mo   12 non-null     int64  
 9   ves_val_mo   12 non-null     int64  
 10  ves_swt_mo   12 non-null     int64  
 11  cnt_val_mo   12 non-null     int64  
 12  cnt_swt_mo   12 non-null     int64  
 13  gen_val_yr   12 non-null     int64  
 14  air_val_yr   12 non-null     int64  
 15  air_swt_yr   12 non-null     int64  
 16  ves_val_yr   12 non-null     int64  
 17  ves_swt_yr   12 non-null     int64  
 18  cnt_val_yr   12 non-null     int64  
 19  cnt_swt_yr   12 n

In [45]:
# Mexico washing machines — Dec 2025 (PORTHS6MM)
df_mex_wash_2512 = df_ocean_routes_2512[
    (df_ocean_routes_2512["cty_code"] == "2010") &
    (df_ocean_routes_2512["commodity"].isin(["845011", "845020"])) &
    (df_ocean_routes_2512["port_full"].isin(coord.us_ports))
].copy()

for col in ["gen_val_mo", "ves_val_mo", "ves_swt_mo"]:
    df_mex_wash_2512[col] = pd.to_numeric(df_mex_wash_2512[col], errors="coerce").fillna(0)

df_mex_wash_2512 = df_mex_wash_2512[df_mex_wash_2512["gen_val_mo"] > 0]

df_mex_wash_2512["gen_swt_mo"] = df_mex_wash_2512["ves_swt_mo"].where(
    df_mex_wash_2512["ves_swt_mo"] > 0,
    df_mex_wash_2512["gen_val_mo"] / washing_machine_price_coeff
)
df_mex_wash_2512["distance"] = df_mex_wash_2512.apply(
    core.compute_distance,
    args=(coord.mexico_start, coord.us_ports, False, None, "port_full"), axis=1)
df_mex_wash_2512["co2"] = df_mex_wash_2512.apply(core.compute_co2, args=["gen_swt_mo"], axis=1)
df_mex_wash_2512.info()

<class 'pandas.DataFrame'>
Index: 9 entries, 824105 to 824264
Data columns (total 24 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    9 non-null      str    
 1   cty_code     9 non-null      str    
 2   dist_unlade  9 non-null      str    
 3   port_unlade  9 non-null      str    
 4   year         9 non-null      str    
 5   month        9 non-null      str    
 6   gen_val_mo   9 non-null      int64  
 7   air_val_mo   9 non-null      int64  
 8   air_swt_mo   9 non-null      int64  
 9   ves_val_mo   9 non-null      int64  
 10  ves_swt_mo   9 non-null      int64  
 11  cnt_val_mo   9 non-null      int64  
 12  cnt_swt_mo   9 non-null      int64  
 13  gen_val_yr   9 non-null      int64  
 14  air_val_yr   9 non-null      int64  
 15  air_swt_yr   9 non-null      int64  
 16  ves_val_yr   9 non-null      int64  
 17  ves_swt_yr   9 non-null      int64  
 18  cnt_val_yr   9 non-null      int64  
 19  cnt_swt_yr   9 non

In [46]:
# Mexico TVs — Dec 2024 (PORTHS6MM)
df_mex_tv_2412 = df_ocean_routes_2412[
    (df_ocean_routes_2412["cty_code"] == "2010") &
    (df_ocean_routes_2412["commodity"].str.startswith("8528")) &
    (df_ocean_routes_2412["port_full"].isin(coord.us_ports))
].copy()

for col in ["gen_val_mo", "ves_val_mo", "ves_swt_mo"]:
    df_mex_tv_2412[col] = pd.to_numeric(df_mex_tv_2412[col], errors="coerce").fillna(0)

df_mex_tv_2412 = df_mex_tv_2412[df_mex_tv_2412["gen_val_mo"] > 0]

df_mex_tv_2412["gen_swt_mo"] = df_mex_tv_2412["ves_swt_mo"].where(
    df_mex_tv_2412["ves_swt_mo"] > 0,
    df_mex_tv_2412["gen_val_mo"] / tv_price_coeff
)
df_mex_tv_2412["distance"] = df_mex_tv_2412.apply(
    core.compute_distance,
    args=(coord.mexico_start, coord.us_ports, False, None, "port_full"), axis=1)
df_mex_tv_2412["co2"] = df_mex_tv_2412.apply(core.compute_co2, args=["gen_swt_mo"], axis=1)
df_mex_tv_2412.info()

<class 'pandas.DataFrame'>
Index: 45 entries, 953265 to 955749
Data columns (total 24 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    45 non-null     str    
 1   cty_code     45 non-null     str    
 2   dist_unlade  45 non-null     str    
 3   port_unlade  45 non-null     str    
 4   year         45 non-null     str    
 5   month        45 non-null     str    
 6   gen_val_mo   45 non-null     int64  
 7   air_val_mo   45 non-null     int64  
 8   air_swt_mo   45 non-null     int64  
 9   ves_val_mo   45 non-null     int64  
 10  ves_swt_mo   45 non-null     int64  
 11  cnt_val_mo   45 non-null     int64  
 12  cnt_swt_mo   45 non-null     int64  
 13  gen_val_yr   45 non-null     int64  
 14  air_val_yr   45 non-null     int64  
 15  air_swt_yr   45 non-null     int64  
 16  ves_val_yr   45 non-null     int64  
 17  ves_swt_yr   45 non-null     int64  
 18  cnt_val_yr   45 non-null     int64  
 19  cnt_swt_yr   45 n

In [47]:
# Mexico TVs — Dec 2025 (PORTHS6MM)
df_mex_tv_2512 = df_ocean_routes_2512[
    (df_ocean_routes_2512["cty_code"] == "2010") &
    (df_ocean_routes_2512["commodity"].str.startswith("8528")) &
    (df_ocean_routes_2512["port_full"].isin(coord.us_ports))
].copy()

for col in ["gen_val_mo", "ves_val_mo", "ves_swt_mo"]:
    df_mex_tv_2512[col] = pd.to_numeric(df_mex_tv_2512[col], errors="coerce").fillna(0)

df_mex_tv_2512 = df_mex_tv_2512[df_mex_tv_2512["gen_val_mo"] > 0]

df_mex_tv_2512["gen_swt_mo"] = df_mex_tv_2512["ves_swt_mo"].where(
    df_mex_tv_2512["ves_swt_mo"] > 0,
    df_mex_tv_2512["gen_val_mo"] / tv_price_coeff
)
df_mex_tv_2512["distance"] = df_mex_tv_2512.apply(
    core.compute_distance,
    args=(coord.mexico_start, coord.us_ports, False, None, "port_full"), axis=1)
df_mex_tv_2512["co2"] = df_mex_tv_2512.apply(core.compute_co2, args=["gen_swt_mo"], axis=1)
df_mex_tv_2512.info()

<class 'pandas.DataFrame'>
Index: 40 entries, 978555 to 980954
Data columns (total 24 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    40 non-null     str    
 1   cty_code     40 non-null     str    
 2   dist_unlade  40 non-null     str    
 3   port_unlade  40 non-null     str    
 4   year         40 non-null     str    
 5   month        40 non-null     str    
 6   gen_val_mo   40 non-null     int64  
 7   air_val_mo   40 non-null     int64  
 8   air_swt_mo   40 non-null     int64  
 9   ves_val_mo   40 non-null     int64  
 10  ves_swt_mo   40 non-null     int64  
 11  cnt_val_mo   40 non-null     int64  
 12  cnt_swt_mo   40 non-null     int64  
 13  gen_val_yr   40 non-null     int64  
 14  air_val_yr   40 non-null     int64  
 15  air_swt_yr   40 non-null     int64  
 16  ves_val_yr   40 non-null     int64  
 17  ves_swt_yr   40 non-null     int64  
 18  cnt_val_yr   40 non-null     int64  
 19  cnt_swt_yr   40 n

In [48]:
df_asian_tv_2512["port_full"].value_counts().sort_index()
TvPortcodes2512 = df_asian_tv_2512["port_full"].value_counts().sort_index()
with pd.option_context('display.max_rows', None,
                       'display.max_columns', None,
                       'display.precision', 3,
                       ):
    print(TvPortcodes2512)

port_full
0401     2
1001     4
1003     9
1102     1
1303     2
1401     5
1601     5
1703     8
1801     2
1816     1
1901     3
2002     1
2704    16
2709    13
2809     2
2811     9
2904     1
3001     6
3002     4
3604     1
4909     2
5201     5
5203     2
5301     5
5310     2
Name: count, dtype: int64


In [49]:
# import requests
# from pathlib import Path

# OUT = Path("data/port_hs6")
# OUT.mkdir(parents=True, exist_ok=True)

# def url(year, month):
#     yy = str(year)[2:]
#     return (f"https://www.census.gov/trade/downloads/{year}"
#             f"/Port/im_hs6_m/PORTHS6MM{yy}{month:02d}.ZIP")

# # Your tariff treatment window
# for year in (2023, 2024, 2025):
#     for month in range(1, 13):
#         u = url(year, month)
#         out_path = OUT / f"PORTHS6MM{str(year)[2:]}{month:02d}.ZIP"
#         if out_path.exists():
#             continue
#         r = requests.get(u, timeout=30)
#         if r.status_code == 200:
#             out_path.write_bytes(r.content)
#             print(f"saved {out_path}")
#         else:
#             print(f"missing or not yet released: {u}")

## Pre-Tariff Baseline — Dec 2022 (PORTHS6MM2212, parallel trends)

In [50]:
_snapshot = Path("data/snapshots/PORTHS6MM2212.parquet")
if _snapshot.exists():
    df_ocean_routes_2212 = pd.read_parquet(_snapshot)
else:
    with zipfile.ZipFile("data/original/PORTHS6MM2212.ZIP") as zf:
        with zf.open(zf.namelist()[0]) as f:
            df_ocean_routes_2212 = pd.read_fwf(
                f,
                colspecs=coord.colspecs_sea,
                names=coord.names_sea,
                dtype={c: str for c in coord.str_cols_sea},
            )
    _snapshot.parent.mkdir(exist_ok=True)
    df_ocean_routes_2212.to_parquet(_snapshot, index=False)
df_ocean_routes_2212["port_full"] = (
    df_ocean_routes_2212["dist_unlade"] + df_ocean_routes_2212["port_unlade"])

In [51]:
df_asian_2212 = df_ocean_routes_2212[
    df_ocean_routes_2212["cty_code"].isin(coord.asian_origins)
].T.drop_duplicates().T
for col in numerical_cols_asian:
    df_asian_2212[col] = pd.to_numeric(df_asian_2212[col])

df_asian_wash_2212 = df_asian_2212[df_asian_2212["commodity"].isin(["845011", "845020"])].copy()
df_asian_wash_2212 = df_asian_wash_2212[df_asian_wash_2212["port_full"].isin(coord.us_ports)]
df_asian_wash_2212 = df_asian_wash_2212[df_asian_wash_2212["ves_swt_mo"] > 0]
df_asian_wash_2212["distance"] = df_asian_wash_2212.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_wash_2212["co2"] = df_asian_wash_2212.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

df_asian_tv_2212 = df_asian_2212[df_asian_2212["commodity"].str.startswith("8528")].copy()
df_asian_tv_2212 = df_asian_tv_2212[df_asian_tv_2212["port_full"].isin(coord.us_ports)]
df_asian_tv_2212 = df_asian_tv_2212[df_asian_tv_2212["ves_swt_mo"] > 0]
df_asian_tv_2212["distance"] = df_asian_tv_2212.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_tv_2212["co2"] = df_asian_tv_2212.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

In [52]:
# Period-specific price coefficients for 2212
_mv = df_ocean_routes_2212[
    (df_ocean_routes_2212["cty_code"] == "2010") &
    (df_ocean_routes_2212["commodity"].isin(["845011", "845020"]))
].copy()
for col in ["ves_val_mo", "ves_swt_mo"]:
    _mv[col] = pd.to_numeric(_mv[col], errors="coerce").fillna(0)
_mv = _mv[_mv["ves_swt_mo"] > 0]
wash_pc_2212 = (_mv["ves_val_mo"].mean() / _mv["ves_swt_mo"].mean()
                    if not _mv.empty else washing_machine_price_coeff)

_tv = df_ocean_routes_2212[
    (df_ocean_routes_2212["cty_code"] == "2010") &
    (df_ocean_routes_2212["commodity"].str.startswith("8528"))
].copy()
for col in ["ves_val_mo", "ves_swt_mo"]:
    _tv[col] = pd.to_numeric(_tv[col], errors="coerce").fillna(0)
_tv = _tv[_tv["ves_swt_mo"] > 0]
tv_pc_2212 = (_tv["ves_val_mo"].mean() / _tv["ves_swt_mo"].mean()
                  if not _tv.empty else tv_price_coeff)

# Mexico washing machines — 2212
df_mex_wash_2212 = df_ocean_routes_2212[
    (df_ocean_routes_2212["cty_code"] == "2010") &
    (df_ocean_routes_2212["commodity"].isin(["845011", "845020"])) &
    (df_ocean_routes_2212["port_full"].isin(coord.us_ports))
].copy()
for col in ["gen_val_mo", "ves_val_mo", "ves_swt_mo"]:
    df_mex_wash_2212[col] = pd.to_numeric(df_mex_wash_2212[col], errors="coerce").fillna(0)
df_mex_wash_2212 = df_mex_wash_2212[df_mex_wash_2212["gen_val_mo"] > 0]
df_mex_wash_2212["gen_swt_mo"] = df_mex_wash_2212["ves_swt_mo"].where(
    df_mex_wash_2212["ves_swt_mo"] > 0, df_mex_wash_2212["gen_val_mo"] / wash_pc_2212)
df_mex_wash_2212["distance"] = df_mex_wash_2212.apply(
    core.compute_distance, args=(coord.mexico_start, coord.us_ports, False, None, "port_full"), axis=1)
df_mex_wash_2212["co2"] = df_mex_wash_2212.apply(core.compute_co2, args=["gen_swt_mo"], axis=1)

# Mexico TVs — 2212
df_mex_tv_2212 = df_ocean_routes_2212[
    (df_ocean_routes_2212["cty_code"] == "2010") &
    (df_ocean_routes_2212["commodity"].str.startswith("8528")) &
    (df_ocean_routes_2212["port_full"].isin(coord.us_ports))
].copy()
for col in ["gen_val_mo", "ves_val_mo", "ves_swt_mo"]:
    df_mex_tv_2212[col] = pd.to_numeric(df_mex_tv_2212[col], errors="coerce").fillna(0)
df_mex_tv_2212 = df_mex_tv_2212[df_mex_tv_2212["gen_val_mo"] > 0]
df_mex_tv_2212["gen_swt_mo"] = df_mex_tv_2212["ves_swt_mo"].where(
    df_mex_tv_2212["ves_swt_mo"] > 0, df_mex_tv_2212["gen_val_mo"] / tv_pc_2212)
df_mex_tv_2212["distance"] = df_mex_tv_2212.apply(
    core.compute_distance, args=(coord.mexico_start, coord.us_ports, False, None, "port_full"), axis=1)
df_mex_tv_2212["co2"] = df_mex_tv_2212.apply(core.compute_co2, args=["gen_swt_mo"], axis=1)

## Pre-Tariff Baseline — Dec 2023 (PORTHS6MM2312, parallel trends)

In [53]:
_snapshot = Path("data/snapshots/PORTHS6MM2312.parquet")
if _snapshot.exists():
    df_ocean_routes_2312 = pd.read_parquet(_snapshot)
else:
    with zipfile.ZipFile("data/original/PORTHS6MM2312.ZIP") as zf:
        with zf.open(zf.namelist()[0]) as f:
            df_ocean_routes_2312 = pd.read_fwf(
                f,
                colspecs=coord.colspecs_sea,
                names=coord.names_sea,
                dtype={c: str for c in coord.str_cols_sea},
            )
    _snapshot.parent.mkdir(exist_ok=True)
    df_ocean_routes_2312.to_parquet(_snapshot, index=False)
df_ocean_routes_2312["port_full"] = (
    df_ocean_routes_2312["dist_unlade"] + df_ocean_routes_2312["port_unlade"])

In [54]:
df_asian_2312 = df_ocean_routes_2312[
    df_ocean_routes_2312["cty_code"].isin(coord.asian_origins)
].T.drop_duplicates().T
for col in numerical_cols_asian:
    df_asian_2312[col] = pd.to_numeric(df_asian_2312[col])

df_asian_wash_2312 = df_asian_2312[df_asian_2312["commodity"].isin(["845011", "845020"])].copy()
df_asian_wash_2312 = df_asian_wash_2312[df_asian_wash_2312["port_full"].isin(coord.us_ports)]
df_asian_wash_2312 = df_asian_wash_2312[df_asian_wash_2312["ves_swt_mo"] > 0]
df_asian_wash_2312["distance"] = df_asian_wash_2312.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_wash_2312["co2"] = df_asian_wash_2312.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

df_asian_tv_2312 = df_asian_2312[df_asian_2312["commodity"].str.startswith("8528")].copy()
df_asian_tv_2312 = df_asian_tv_2312[df_asian_tv_2312["port_full"].isin(coord.us_ports)]
df_asian_tv_2312 = df_asian_tv_2312[df_asian_tv_2312["ves_swt_mo"] > 0]
df_asian_tv_2312["distance"] = df_asian_tv_2312.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_tv_2312["co2"] = df_asian_tv_2312.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

In [55]:
# Period-specific price coefficients for 2312
_mv = df_ocean_routes_2312[
    (df_ocean_routes_2312["cty_code"] == "2010") &
    (df_ocean_routes_2312["commodity"].isin(["845011", "845020"]))
].copy()
for col in ["ves_val_mo", "ves_swt_mo"]:
    _mv[col] = pd.to_numeric(_mv[col], errors="coerce").fillna(0)
_mv = _mv[_mv["ves_swt_mo"] > 0]
wash_pc_2312 = (_mv["ves_val_mo"].mean() / _mv["ves_swt_mo"].mean()
                    if not _mv.empty else washing_machine_price_coeff)

_tv = df_ocean_routes_2312[
    (df_ocean_routes_2312["cty_code"] == "2010") &
    (df_ocean_routes_2312["commodity"].str.startswith("8528"))
].copy()
for col in ["ves_val_mo", "ves_swt_mo"]:
    _tv[col] = pd.to_numeric(_tv[col], errors="coerce").fillna(0)
_tv = _tv[_tv["ves_swt_mo"] > 0]
tv_pc_2312 = (_tv["ves_val_mo"].mean() / _tv["ves_swt_mo"].mean()
                  if not _tv.empty else tv_price_coeff)

# Mexico washing machines — 2312
df_mex_wash_2312 = df_ocean_routes_2312[
    (df_ocean_routes_2312["cty_code"] == "2010") &
    (df_ocean_routes_2312["commodity"].isin(["845011", "845020"])) &
    (df_ocean_routes_2312["port_full"].isin(coord.us_ports))
].copy()
for col in ["gen_val_mo", "ves_val_mo", "ves_swt_mo"]:
    df_mex_wash_2312[col] = pd.to_numeric(df_mex_wash_2312[col], errors="coerce").fillna(0)
df_mex_wash_2312 = df_mex_wash_2312[df_mex_wash_2312["gen_val_mo"] > 0]
df_mex_wash_2312["gen_swt_mo"] = df_mex_wash_2312["ves_swt_mo"].where(
    df_mex_wash_2312["ves_swt_mo"] > 0, df_mex_wash_2312["gen_val_mo"] / wash_pc_2312)
df_mex_wash_2312["distance"] = df_mex_wash_2312.apply(
    core.compute_distance, args=(coord.mexico_start, coord.us_ports, False, None, "port_full"), axis=1)
df_mex_wash_2312["co2"] = df_mex_wash_2312.apply(core.compute_co2, args=["gen_swt_mo"], axis=1)

# Mexico TVs — 2312
df_mex_tv_2312 = df_ocean_routes_2312[
    (df_ocean_routes_2312["cty_code"] == "2010") &
    (df_ocean_routes_2312["commodity"].str.startswith("8528")) &
    (df_ocean_routes_2312["port_full"].isin(coord.us_ports))
].copy()
for col in ["gen_val_mo", "ves_val_mo", "ves_swt_mo"]:
    df_mex_tv_2312[col] = pd.to_numeric(df_mex_tv_2312[col], errors="coerce").fillna(0)
df_mex_tv_2312 = df_mex_tv_2312[df_mex_tv_2312["gen_val_mo"] > 0]
df_mex_tv_2312["gen_swt_mo"] = df_mex_tv_2312["ves_swt_mo"].where(
    df_mex_tv_2312["ves_swt_mo"] > 0, df_mex_tv_2312["gen_val_mo"] / tv_pc_2312)
df_mex_tv_2312["distance"] = df_mex_tv_2312.apply(
    core.compute_distance, args=(coord.mexico_start, coord.us_ports, False, None, "port_full"), axis=1)
df_mex_tv_2312["co2"] = df_mex_tv_2312.apply(core.compute_co2, args=["gen_swt_mo"], axis=1)

# CO2 Calculations(transportation)

Now that we have the three necessary components - ton-km CO2 factor, weight of imports and distance of imports, then we can calculate the carbon footprint of transportation. This is only one step, as we also need to calculate CO2 of manufacturing(grid intensity x energy spent on manufacturing) in order to understand the full extent of the carbon footprint.

CO2, in this case, is measured in grams. For now, we do this only for 5 countries(China, Vietnam, South Korea, India, Mexico) and only for the month of January 2025, using the datasets taken from census.gov website.

In [56]:
# df_asian_routes_wash["co2"] = df_asian_routes_wash["vessel_swt_mo"]*df_asian_routes_wash["distance"]*ton_km[df_asian_routes_wash["cty_code"]]
df_asian_tv_2412["co2"] = df_asian_tv_2412.apply(core.compute_co2, args = ["ves_swt_mo"], axis=1)


In [57]:
df_asian_tv_2412.head()

,commodity,cty_code,dist_unlade,port_unlade,year,month,gen_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,gen_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr,port_full,distance,co2
953212,852849,5700,27,09,2024,12,136068,0,0,136068,...,174835,0,0,174835,27408,174835,27408,2709,10674.961095,8.998138e+05
953794,852852,5520,17,03,2024,12,626316,0,0,626316,...,2608368,0,0,2608368,79266,2608368,79266,1703,20949.030244,3.620474e+06
953797,852852,5520,27,04,2024,12,35278490,0,0,35278490,...,425218271,0,0,425218271,23321912,422194514,23231547,2704,13525.167789,2.115113e+08
953798,852852,5520,27,09,2024,12,1250476,0,0,1250476,...,52803271,0,0,52803271,2479649,52803271,2479649,2709,13525.167789,3.925464e+06
953803,852852,5520,30,02,2024,12,1117640,0,0,1117640,...,3321417,0,0,3321417,114877,3321417,114877,3002,12411.976275,3.666411e+06


In [58]:
df_asian_tv_2412.info()

<class 'pandas.DataFrame'>
Index: 116 entries, 953212 to 955952
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    116 non-null    object 
 1   cty_code     116 non-null    object 
 2   dist_unlade  116 non-null    object 
 3   port_unlade  116 non-null    object 
 4   year         116 non-null    object 
 5   month        116 non-null    object 
 6   gen_val_mo   116 non-null    int64  
 7   air_val_mo   116 non-null    int64  
 8   air_swt_mo   116 non-null    int64  
 9   ves_val_mo   116 non-null    int64  
 10  ves_swt_mo   116 non-null    int64  
 11  cnt_val_mo   116 non-null    int64  
 12  cnt_swt_mo   116 non-null    int64  
 13  gen_val_yr   116 non-null    object 
 14  air_val_yr   116 non-null    object 
 15  air_swt_yr   116 non-null    object 
 16  ves_val_yr   116 non-null    object 
 17  ves_swt_yr   116 non-null    object 
 18  cnt_val_yr   116 non-null    object 
 19  cnt_swt_yr   116

# Statistics

Now we use the difference-in-difference method to evaluate the extent, to which the tariffs have changed the CO2 footprint. We compare the differences in the control group and the treatment group. 

What will be the control group in our case? 

The control group will be "commodity", more specifically TV sets. TVs have mostly the same demand as washing machines, since both are bought together when people move to a new home. However, unlike the washing machines, TV sets don't fall under the Liberation Day tariffs and steel tariffs of 2025, which is due to goods with semiconductors being exempt from tariffs. These factors make TVs a good control group, since they're similar to washing machines in most key regards, except for the thing that we try to measure.

However, TV sets are lighter than washing machines, so CO2 change scales differently. One way to fix this is to normalize CO2 data by calculating CO2/kg of commodity, rather than total CO2. 


This code sets up a DiD model and runs an OLS regression combining **both Mexican and Asian data**. Mexican observations are grouped by destination state; Asian observations are grouped by origin country and destination port.

Then it creates a long table, in which each row is a combination of country, date and product(20 rows in total). Each row has total weight of products imported, total CO2 released and weighted average distance. Each of the aforementioned

In [59]:
def _tag(df, commodity, period, weight_col="ves_swt_mo"):
    out = df[["cty_code", weight_col, "co2", "distance", "port_full"]].copy()
    out = out.rename(columns={weight_col: "weight"})
    out["commodity"] = commodity
    out["period"] = period
    return out

df_sea_all = pd.concat([
    _tag(df_asian_wash_2412, "washing_machine", "2412"),
    _tag(df_asian_wash_2512, "washing_machine", "2512"),
    _tag(df_asian_tv_2412,   "tv",              "2412"),
    _tag(df_asian_tv_2512,   "tv",              "2512"),
    _tag(df_mex_wash_2412,  "washing_machine", "2412", weight_col="gen_swt_mo"),
    _tag(df_mex_wash_2512,  "washing_machine", "2512", weight_col="gen_swt_mo"),
    _tag(df_mex_tv_2412,    "tv",              "2412", weight_col="gen_swt_mo"),
    _tag(df_mex_tv_2512,    "tv",              "2512", weight_col="gen_swt_mo"),
    _tag(df_asian_wash_2212, "washing_machine", "2212"),
    _tag(df_asian_wash_2312, "washing_machine", "2312"),
    _tag(df_asian_tv_2212,   "tv",              "2212"),
    _tag(df_asian_tv_2312,   "tv",              "2312"),
    _tag(df_mex_wash_2212,  "washing_machine", "2212", weight_col="gen_swt_mo"),
    _tag(df_mex_wash_2312,  "washing_machine", "2312", weight_col="gen_swt_mo"),
    _tag(df_mex_tv_2212,    "tv",              "2212", weight_col="gen_swt_mo"),
    _tag(df_mex_tv_2312,    "tv",              "2312", weight_col="gen_swt_mo"),
], ignore_index=True)

df_sea_all["dist_x_weight"] = df_sea_all["distance"] * df_sea_all["weight"]

df_sea_agg = df_sea_all.groupby(["cty_code", "commodity", "period", "port_full"]).agg(
    total_weight=("weight", "sum"),
    total_co2=("co2", "sum"),
    _dist_x_wt=("dist_x_weight", "sum"),
).reset_index()

df_sea_agg["avg_distance"] = df_sea_agg["_dist_x_wt"] / df_sea_agg["total_weight"]
df_sea_agg = df_sea_agg.drop(columns="_dist_x_wt")
df_sea_agg

,cty_code,commodity,period,port_full,total_weight,total_co2,avg_distance
0,2010,tv,2212,0712,10662.023301,3.128244e+06,3667.507268
1,2010,tv,2212,0901,2145.504274,7.260091e+05,4229.827612
2,2010,tv,2212,1003,75.000000,2.287262e+04,3812.103580
3,2010,tv,2212,2002,1503.297872,1.741133e+05,1447.761106
4,2010,tv,2212,2301,286571.030995,6.573317e+06,286.722871
...,...,...,...,...,...,...,...
500,5800,washing_machine,2512,3001,18284.000000,7.123308e+05,8657.609915
501,5800,washing_machine,2512,3002,93126.000000,3.642960e+06,8693.025431
502,5800,washing_machine,2512,4909,26498.000000,2.039909e+06,17107.446556
503,5800,washing_machine,2512,5201,3204.000000,2.527945e+05,17533.255231


In [60]:
mex_slice = df_sea_agg[df_sea_agg["cty_code"] == "2010"]
mex_slice.head()

,cty_code,commodity,period,port_full,total_weight,total_co2,avg_distance
0,2010,tv,2212,0712,10662.023301,3.128244e+06,3667.507268
1,2010,tv,2212,0901,2145.504274,7.260091e+05,4229.827612
2,2010,tv,2212,1003,75.000000,2.287262e+04,3812.103580
3,2010,tv,2212,2002,1503.297872,1.741133e+05,1447.761106
4,2010,tv,2212,2301,286571.030995,6.573317e+06,286.722871


In [61]:
mex_slice_check = mex_slice.copy()
mex_slice_check["co2_intensity"] = mex_slice_check["total_co2"] / mex_slice_check["total_weight"]
mex_slice_check.head()

,cty_code,commodity,period,port_full,total_weight,total_co2,avg_distance,co2_intensity
0,2010,tv,2212,0712,10662.023301,3.128244e+06,3667.507268,293.400581
1,2010,tv,2212,0901,2145.504274,7.260091e+05,4229.827612,338.386209
2,2010,tv,2212,1003,75.000000,2.287262e+04,3812.103580,304.968286
3,2010,tv,2212,2002,1503.297872,1.741133e+05,1447.761106,115.820889
4,2010,tv,2212,2301,286571.030995,6.573317e+06,286.722871,22.937830


In [62]:
mex_slice_check.groupby(["port_full", "commodity"])["co2_intensity"].std().describe()

count    3.300000e+01
mean     1.142665e-14
std      1.419929e-14
min      0.000000e+00
25%      0.000000e+00
50%      3.552714e-15
75%      1.834613e-14
max      4.019437e-14
Name: co2_intensity, dtype: float64

In [63]:
_land = df_ocean_routes_2412[df_ocean_routes_2412["cty_code"] == "2010"].copy()
_land["gen_val_mo"] = pd.to_numeric(_land["gen_val_mo"], errors="coerce").fillna(0)
print("rows with gen_val_mo > 0:", (_land["gen_val_mo"] > 0).sum(), "of", len(_land))

rows with gen_val_mo > 0: 14698 of 33555


In [64]:
import statsmodels.api as sm

df_did = df_sea_agg[df_sea_agg["total_weight"] > 0].copy()
df_did["co2_intensity"] = df_did["total_co2"] / df_did["total_weight"]
df_did["treated"] = (df_did["commodity"] == "washing_machine").astype(int)
df_did["post"]    = (df_did["period"]    == "2512").astype(int)
df_did["s"] = df_did["cty_code"].map(coord.tariff_rate) * df_did["treated"]
df_did["treated_x_post"] = df_did["s"] * df_did["post"]

df_did[["cty_code", "port_full", "co2_intensity", "treated", "post", "treated_x_post"]].sort_values("co2_intensity", ascending=False).head(10)

,cty_code,port_full,co2_intensity,treated,post,treated_x_post
91,2010,3310,1201.052679,0,1,0.00
70,2010,3126,562.331561,0,0,0.00
16,2010,3126,562.331561,0,0,0.00
103,2010,4909,374.910450,1,0,0.00
97,2010,4909,374.910450,0,1,0.00
110,2010,4909,374.910450,1,0,0.00
124,2010,4909,374.910450,1,1,0.04
47,2010,4909,374.910450,0,0,0.00
118,2010,4909,374.910450,1,0,0.00
21,2010,4909,374.910450,0,0,0.00


In [65]:
df_sea_all.head()

,cty_code,weight,co2,distance,port_full,commodity,period,dist_x_weight
0,5520,223317.0,3.130660e+07,20027.006554,1003,washing_machine,2412,4.472371e+09
1,5520,2683.0,3.866830e+05,20589.054455,1303,washing_machine,2412,5.524043e+07
2,5520,4488.0,6.581347e+05,20949.030244,1703,washing_machine,2412,9.401925e+07
3,5520,13663.0,2.034897e+06,21276.406623,1801,washing_machine,2412,2.906995e+08
4,5520,89496.0,8.473139e+06,13525.167789,2704,washing_machine,2412,1.210448e+09


In [66]:
# CO2 intensity variation by origin country — a non-zero std means the DiD has signal to work with
df_did.groupby(["cty_code", "commodity"])["co2_intensity"].std().describe()

count      9.000000
mean      47.735903
std       55.534898
min       11.957109
25%       17.895028
50%       19.893426
75%       27.262073
max      152.854105
Name: co2_intensity, dtype: float64

A sanity check. If std = 0, then tinkering with it is useless. For land-only dataset(Mexico) it used to be 0. For the current united dataset, it's 38.

In [67]:
sea_check = df_sea_agg.copy()
sea_check["co2_intensity"] = sea_check["total_co2"] / sea_check["total_weight"]

sea_check.groupby(["port_full", "commodity"])["co2_intensity"].std().describe()

count    7.300000e+01
mean     2.451903e+01
std      3.642518e+01
min      0.000000e+00
25%      3.552714e-15
50%      3.191601e+00
75%      3.402201e+01
max      1.619634e+02
Name: co2_intensity, dtype: float64

In [68]:
X = sm.add_constant(df_did[["treated", "post", "treated_x_post"]])
model = sm.OLS(df_did["co2_intensity"], X, missing="drop").fit(cov_type="HC3")
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:          co2_intensity   R-squared:                       0.022
Model:                            OLS   Adj. R-squared:                  0.016
Method:                 Least Squares   F-statistic:                     5.470
Date:                Fri, 14 Aug 2026   Prob (F-statistic):            0.00105
Time:                        21:55:03   Log-Likelihood:                -2999.8
No. Observations:                 505   AIC:                             6008.
Df Residuals:                     501   BIC:                             6025.
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const            120.3470      6.209     19.

In [69]:
# Event-study parallel-trends test (reference period: 2412)
df_did["d2212"] = (df_did["period"] == "2212").astype(int)
df_did["d2312"] = (df_did["period"] == "2312").astype(int)

df_did["treat_x_d2212"] = df_did["s"] * df_did["d2212"]
df_did["treat_x_d2312"] = df_did["s"] * df_did["d2312"]

X_es = sm.add_constant(df_did[[
    "treated", "d2212", "d2312", "post",
    "treat_x_d2212", "treat_x_d2312", "treated_x_post"
]])
model_es = sm.OLS(df_did["co2_intensity"], X_es, missing="drop").fit(cov_type="HC3")
print(model_es.summary())

print("\nJoint pre-trend F-test (treat_x_d2212 = treat_x_d2312 = 0):")
print(model_es.f_test("treat_x_d2212 = 0, treat_x_d2312 = 0"))

                            OLS Regression Results                            
Dep. Variable:          co2_intensity   R-squared:                       0.023
Model:                            OLS   Adj. R-squared:                  0.010
Method:                 Least Squares   F-statistic:                     3.119
Date:                Fri, 14 Aug 2026   Prob (F-statistic):            0.00313
Time:                        21:55:03   Log-Likelihood:                -2999.5
No. Observations:                 505   AIC:                             6015.
Df Residuals:                     497   BIC:                             6049.
Df Model:                           7                                         
Covariance Type:                  HC3                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const            118.7885     11.428     10.

Having run the OLS regression on the difference-in-differences model, we found that for all countries' imports, there is no difference in the CO2 before and after the tariffs. 

Why could that be? I know that for land transportation to the USA, the number of trade routes is more limited compared to ocean routes, so there is little that Mexico can do to change their trade routes in the aftermath of tariffs. But how much did the number of imported goods change?

Also, how do I interpret those other statistics? F-statistic, log-likelihood, AIC/BIC, Durbin-Watson, et cetera.

## Sensitivity Analysis
An addendum for justifying the job.

In [70]:
import panel
# This code creates a sensitivity analysis of tariff rates and steel content
# Pre-period: no tariffs in force. Post-period: rates from Table 1.
RECIPROCAL = {
    ("China",       "2024-12"): 0.00,
    ("Vietnam",     "2024-12"): 0.00,
    ("South Korea", "2024-12"): 0.00,
    ("China",       "2025-12"): 0.20,   # verify: reciprocal + border
    ("Vietnam",     "2025-12"): 0.20,
    ("South Korea", "2025-12"): 0.15
    }

panel["r"] = panel.apply(
    lambda row: RECIPROCAL[(row["origin"], row["period"])], axis=1
)

ModuleNotFoundError: No module named 'panel'

In [ ]:
# this code puts the dose function in Python terms
S232_RATE = 0.50

def build_dose(df, sigma):
    """Differential ad valorem burden, washers vs TVs.
    Zero for TVs (not steel derivatives) and zero pre-tariff."""
    is_washer = (df["commodity"] == "washer")
    post = (df["period"] == "2025-12")
    return sigma * (S232_RATE - df["r"]) * is_washer * post

In [ ]:
import statsmodels.api as sm
import pandas as pd


SIGMA_GRID = [0.15, 0.25, 0.40]
rows = []

# this code loops over sigma values(steel content) and collects results
for sigma in SIGMA_GRID:
    d = panel.copy()
    d["dose"] = build_dose(d, sigma)
    d["post"] = (d["period"] == "2025-12").astype(int)

    X = sm.add_constant(d[["dose", "post"]])
    fit = sm.OLS(d["co2_intensity"], X).fit(cov_type="HC3")

    lo, hi = fit.conf_int().loc["dose"]
    rows.append({
        "sigma": sigma,
        "delta": fit.params["dose"],
        "se":    fit.bse["dose"],
        "ci_lo": lo,
        "ci_hi": hi,
        "p":     fit.pvalues["dose"],
    })

results = pd.DataFrame(rows)
print(results.round(4))

In [ ]:
# final sanity check - invariance
assert results["p"].nunique() == 1, "p-values should be identical across sigma"
print("delta ratio 0.15/0.40:", results.loc[0,"delta"] / results.loc[2,"delta"])

In [ ]:
# export the results to Latex
tex = results.to_latex(
    index=False,
    float_format="%.3f",
    columns=["sigma", "delta", "ci_lo", "ci_hi", "p"],
    header=["$\\sigma$", "$\\hat{\\delta}$", "CI low", "CI high", "$p$"],
    escape=False,
    caption="Sensitivity of the DiD estimate to assumed steel content share.",
    label="tab:sigma_sensitivity",
)
with open("outputs/sigma_sensitivity.tex", "w") as f:
    f.write(tex)